# Ignitions for ArcGIS Online

Packages the Larimer ignition products from `IgnitionDensity.ipynb` into **one GeoPackage**, published as
**one hosted feature layer**. Everything else in the web map is streamed from public services.

| Layer | Source | In this GeoPackage? |
|---|---|---|
| `ignitions` — points, style by `size_class` | FPA-FOD 2013–2024 (public services stop at 2020) | yes |
| `density_human`, `density_lightning` — classed polygons | IgnitionDensity GeoTIFFs, same classes as the figures | yes |
| `share_human` — classed polygons | IgnitionDensity GeoTIFF | yes |
| `counties` — outlines | TIGER 2024 | yes |
| Fire perimeters, all years | NIFC `InterAgencyFirePerimeterHistory_All_Years_View` | no — stream |
| Fire perimeters, current year | WFIGS `WFIGS_Interagency_Perimeters` | no — stream |
| Current-year incident points | WFIGS `WFIGS_Incident_Locations_Current` | no — stream |

Density surfaces go up as polygons, not rasters: hosted imagery needs the Image extension (or credits), while
polygons are small, draw like the figures, and give a readable pop-up (*0.1–0.3 ignitions per km²*).

**Run order:** on the Mac, run `IgnitionDensity.ipynb` (it writes the GeoTIFFs), then §1–3 here.
Publish from ArcGIS Pro on the VM with `IgnitionAGOL_Publish.ipynb` — Pro's Python has no `fb_tools`.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import rasterio.features
import shapely

import fb_tools
from fb_tools.fuelscape import load_fod

REPO      = Path(fb_tools.__file__).resolve().parents[1]
BOX       = Path.home() / "Library/CloudStorage/Box-Box/MCC"
FOD_FP    = BOX / "data/RDS/FPA_FOD/Data/FPA_FOD_20260615.gpkg"
COUNTY_FP = BOX / "data/boundaries/political/TIGER/tl_2024_us_county/tl_2024_us_county.shp"

# Must match 00e.
AOI_NAME    = "larimer"
FOCUS       = {"STATEFP": "08", "NAME": "Larimer"}
YEARS       = (2013, 2024)
RX_PATTERN  = r"\bRX\b|PRESCRIB|\bPILES\b|PILE BURN"
CRS         = "EPSG:26913"
PRODUCT     = "all_90m_adapt2.5km"
CLASS_EDGES = [0.003, 0.01, 0.03, 0.1, 0.3, 1, 3]   # ignitions per km² over YEARS — the figures' classes

GRID_DIR = REPO / "data/spatial/mod/ignition_density" / AOI_NAME / PRODUCT
GPKG     = REPO / "data/spatial/mod/ignition_density" / AOI_NAME / "agol" / f"{AOI_NAME}_ignitions_agol.gpkg"
OUT_CRS  = "EPSG:4326"
SIMPLIFY_M = 45      # half a 90 m cell: removes the stair steps, shared edges stay shared

## 1 · Ignition points

In [ ]:
counties = gpd.read_file(COUNTY_FP, columns=["STATEFP", "NAME"]).to_crs(CRS)
focus = counties[(counties.STATEFP == FOCUS["STATEFP"]) & (counties.NAME == FOCUS["NAME"])]
aoi = counties[counties.intersects(focus.union_all())].reset_index(drop=True)

fod = load_fod(FOD_FP, aoi=aoi, years=YEARS).to_crs(CRS)
fod = fod[~fod.FIRE_NAME.fillna("").str.upper().str.contains(RX_PATTERN, regex=True)]
fod = gpd.sjoin(fod, aoi[["geometry"]], predicate="within").drop(columns="index_right")

SIZE_CLASSES = {"A": "0–0.25 acres", "B": "0.26–9.9 acres", "C": "10–99 acres", "D": "100–299 acres",
                "E": "300–999 acres", "F": "1,000–4,999 acres", "G": "5,000+ acres"}
CAUSE_GROUP = {"human": "Human-caused", "natural": "Lightning-caused", "unknown": "Cause unknown"}
NO_CAUSE = "No specific cause recorded"
GENERAL_CAUSE = {"Missing data/not specified/undetermined": NO_CAUSE, "Other causes": NO_CAUSE,
                 "Power generation/transmission/distribution": "Power lines and utilities",
                 "Arson/incendiarism": "Arson", "Natural": "Lightning"}

name = fod.FIRE_NAME.fillna("").str.strip().str.title()
ignitions = gpd.GeoDataFrame({
    "fod_id":     fod.FOD_ID,
    "fire_name":  name.mask(name.isin(["", "Unknown"]), "Unnamed"),
    "year":       fod.FIRE_YEAR.astype("int32"),
    "discovered": fod.DISCOVERY_DATE.dt.tz_localize(None),
    "acres":      fod.FIRE_SIZE.round(2),
    "size_class": fod.FIRE_SIZE_CLASS,
    "size_label": fod.FIRE_SIZE_CLASS.map(SIZE_CLASSES),
    "cause":      fod.cause.map(CAUSE_GROUP),
    "cause_detail": fod.NWCG_GENERAL_CAUSE.replace(GENERAL_CAUSE).fillna(NO_CAUSE),
}, geometry=fod.geometry, crs=CRS).sort_values("acres").reset_index(drop=True)   # big fires drawn last

print(f"{len(ignitions):,} ignitions, {YEARS[0]}–{YEARS[1]}")
pd.crosstab(ignitions.size_label, ignitions.cause, margins=True).loc[list(SIZE_CLASSES.values()) + ["All"]]

## 2 · Density surfaces → classed polygons

Each GeoTIFF is cut at the figure classes, traced to polygons, dissolved to one feature per class, and
simplified as a coverage (no gaps or slivers between neighbouring classes). `class_id` orders the classes for a
*Unique symbols* renderer; `label` is the pop-up text.

In [ ]:
def classed_polygons(tif, edges, labels):
    with rasterio.open(tif) as src:
        a, transform, crs = src.read(1), src.transform, src.crs
    valid = np.isfinite(a)
    cls = np.digitize(np.where(valid, a, 0), edges).astype("int16")
    shapes = rasterio.features.shapes(cls, mask=valid, transform=transform)
    gdf = gpd.GeoDataFrame([{"class_id": int(v), "geometry": shapely.geometry.shape(g)} for g, v in shapes], crs=crs)
    gdf = gdf.dissolve("class_id").reset_index()
    gdf["geometry"] = shapely.coverage_simplify(gdf.geometry.values, SIMPLIFY_M)
    gdf.insert(1, "label", gdf.class_id.map(dict(enumerate(labels))))
    return gdf


def range_labels(edges, unit, fmt="{:g}"):
    f = [fmt.format(e) for e in edges]
    return [f"under {f[0]} {unit}"] + [f"{lo}–{hi} {unit}" for lo, hi in zip(f[:-1], f[1:])] + [f"{f[-1]}+ {unit}"]


DENSITY_LABELS = range_labels(CLASS_EDGES, "ignitions per km²")
SHARE_EDGES = np.round(np.arange(0.1, 1.0, 0.1), 1)
SHARE_LABELS = [f"{int(lo * 100)}–{int(hi * 100)}% human-caused" for lo, hi in zip([0, *SHARE_EDGES], [*SHARE_EDGES, 1])]

tif = lambda var: GRID_DIR / f"idg_{AOI_NAME}_{PRODUCT}_{var}.tif"
density_human     = classed_polygons(tif("human"), CLASS_EDGES, DENSITY_LABELS)
density_lightning = classed_polygons(tif("natural"), CLASS_EDGES, DENSITY_LABELS)
share_human       = classed_polygons(tif("human_share"), SHARE_EDGES, SHARE_LABELS)

for n, g in [("density_human", density_human), ("density_lightning", density_lightning), ("share_human", share_human)]:
    print(f"{n:18s} {len(g):2d} classes, {shapely.get_num_coordinates(g.geometry.values).sum():>9,} vertices")

In [ ]:
# Quick look: should match the IgnitionDensity figure panels (the share map's blanks are 'too few ignitions to tell').
fig, axs = plt.subplots(1, 3, figsize=(15, 5.5), constrained_layout=True)
for ax, (title, g, cmap) in zip(axs, [("Human-caused", density_human, "Oranges"),
                                      ("Lightning-caused", density_lightning, "Blues"),
                                      ("Share human-caused", share_human, "RdBu_r")]):
    g.plot(ax=ax, column="class_id", cmap=cmap, vmin=0, vmax=g.class_id.max() if "Share" in title else 7)
    aoi.boundary.plot(ax=ax, color="0.3", lw=0.5)
    ax.set_title(title, loc="left")
    ax.set_axis_off()

## 3 · Write the GeoPackage

In [ ]:
LAYERS = {"ignitions": ignitions, "density_human": density_human, "density_lightning": density_lightning,
          "share_human": share_human, "counties": aoi.assign(focus=aoi.NAME.eq(FOCUS["NAME"]))[["NAME", "focus", "geometry"]]}

GPKG.parent.mkdir(parents=True, exist_ok=True)
GPKG.unlink(missing_ok=True)
for layer, gdf in LAYERS.items():
    gdf.to_crs(OUT_CRS).to_file(GPKG, layer=layer, driver="GPKG")
print(f"{GPKG.relative_to(REPO)}  ({GPKG.stat().st_size / 1e6:.1f} MB)")
print(gpd.list_layers(GPKG).to_string(index=False))

## 4 · Publish

Open **`IgnitionAGOL_Publish.ipynb`** in ArcGIS Pro on the VM and *Run All*. It needs only Pro's Python, not `fb_tools` — the same split as pods-atlas (`00POD_Summary` builds, `01AGOL_Prep` publishes in Pro). That notebook also has the browser-upload steps, layer styling, and the streamed-layer URLs.